In [1]:
!pip install scikit-learn
!pip install transformers==4.38.2 datasets==2.16.1 tokenizers==0.15.2 accelerate==0.27.2 lime evaluate openpyxl
!pip uninstall -y numpy
!pip install numpy==1.26.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 212.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 283.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 425.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.3
    Uninstalling numpy-1.26.3:
      Successfully uninstalled numpy-1.26.3

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 181.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 282.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 346.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 151.7 MB

In [2]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix, precision_recall_curve, auc
from transformers import AutoTokenizer, Trainer, TrainingArguments, BertPreTrainedModel, XLMRobertaPreTrainedModel
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers import RobertaTokenizer, RobertaForSequenceClassification,XLMRobertaConfig,XLMRobertaModel
from transformers import AutoModelForSequenceClassification, BertModel, BertConfig
from lime.lime_text import LimeTextExplainer
import torch, torch.nn as nn
from datasets import Dataset
import evaluate, gc, time, numpy as np, pandas as pd, datetime, pickle, csv, re, platform, os, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from transformers import  AutoModelForSequenceClassification, TrainerCallback, PretrainedConfig
import matplotlib.pyplot as plt
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def r2(x):
    if x == "" or x is None:
        return ""
    return round(float(x), 2)

def parse_vector_column(vec):
    if pd.isna(vec):
        return []
    if isinstance(vec, str):
        vec = vec.strip()
        if not vec:
            return []
        return [float(x) for x in vec.split()]
    return [float(x) for x in vec]

def tokenize_and_align_rationale(examples):
    tokenized = tokenizer( examples[xcolumn], truncation=True,
        padding="max_length", max_length=MAX_LENGTH, return_attention_mask=True )
    new_rationale_targets = []
    for i in range(len(examples[xcolumn])):
        rationale = examples["rationale"][i]
        word_ids = tokenized.word_ids(batch_index=i)
        rationale_vec = []
        for wid in word_ids:
            if wid is None or wid >= len(rationale):
                rationale_vec.append(-100)
            else:
                rationale_vec.append(float(rationale[wid]))
        new_rationale_targets.append(rationale_vec)
    tokenized["rationale_targets"] = new_rationale_targets
    return tokenized

In [3]:
MAX_LENGTH = 128
modelname = "roberta"
num_labels = 1
modelpath = "./models/"
N_EPOCHS = 5
lamda = 0.1

file_path = "/workspace/Data/UOLDRationales-24k-V2.xlsx"
result_path =  "/workspace/Results/T1-M6-Result.csv"
LOCAL_OUT = f"/workspace/lc_models/{modelname}" 
save_dir = "/workspace/ft_models/t1-m6-roberta-rat"
MODEL_NAME = "/workspace/models/xlm-roberta-base"
SAVE_result = "/workspace/Results/t1-m6-results_lime.csv"

df = pd.read_excel(file_path, engine='openpyxl')
df.reset_index( drop=True, inplace=True )
df.Tweet = df.Tweet.astype( 'str' )
df.loc[df['Tag'] == 2, 'Tag'] = 1
df.loc[df['Tag'] == 3, 'Tag'] = 1
df.loc[df['Tag'] == 4, 'Tag'] = 1
print(df.columns, df.shape)
xcolumn = "Tweet"
ycolumn = "Tag"
rationale_column = "Vector"

print(df[ycolumn].value_counts().sort_index())
print(datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
df = df[[xcolumn, ycolumn, rationale_column]].dropna()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gc.collect()
torch.cuda.empty_cache()

df['rationale'] = df[rationale_column].apply(parse_vector_column)
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df[ycolumn], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df[ycolumn],random_state=42)
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

columns_to_remove = [col for col in train_dataset.column_names if col not in [xcolumn, "rationale", "Tag"]]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
assert tokenizer.is_fast, "Offset mapping requires a fast tokenizer"

special_token_ids = set(tokenizer.all_special_ids)
# APPLY TOKENIZATION + RATIONALE ALIGNMENT
train_dataset = train_dataset.map(tokenize_and_align_rationale, batched=True, remove_columns=columns_to_remove)
val_dataset   = val_dataset.map(tokenize_and_align_rationale, batched=True, remove_columns=columns_to_remove)
test_dataset  = test_dataset.map(tokenize_and_align_rationale, batched=True, remove_columns=columns_to_remove)

# Rename label column
train_dataset = train_dataset.rename_column(ycolumn, "labels")
val_dataset   = val_dataset.rename_column(ycolumn, "labels")
test_dataset  = test_dataset.rename_column(ycolumn, "labels")

# Set format for PyTorch
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels", "rationale_targets"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels", "rationale_targets"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels", "rationale_targets"])

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple): logits = logits[0]
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    return { "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_binary": f1_score(labels, preds, average="binary", zero_division=0), }


Index(['Unnamed: 0.1', 'Unnamed: 0', 'TweetId', 'Tweet', 'Vector', 'Words',
       'TweetText', 'TweetSpaceInserted', 'Tag'],
      dtype='str') (24063, 9)
Tag
0    15404
1     8659
Name: count, dtype: int64
2026-03-04 07:35:56


Map:   0%|          | 0/17325 [00:00<?, ? examples/s]

Map:   0%|          | 0/1925 [00:00<?, ? examples/s]

Map:   0%|          | 0/4813 [00:00<?, ? examples/s]

In [4]:
class RebertaExplain(XLMRobertaPreTrainedModel):
    config_class = XLMRobertaConfig
    base_model_prefix = "roberta"
    _supports_experts = False
    all_tied_weights_keys = {}
    def __init__(self, config):
        super().__init__(config)
        self.config = config
        self.num_labels = config.num_labels
        self.roberta = XLMRobertaModel(config)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 1)
        self.classifier.weight.data.normal_(mean=0.0, std=0.02)
        self.classifier.bias.data.zero_()
        self.register_buffer("pos_weight", torch.tensor([1.0], dtype=torch.float32))
        self.post_init()
    def forward(self, input_ids=None, attention_mask=None, labels=None, rationale_targets=None, output_attentions=None, **kwargs):
        if output_attentions is None:
            output_attentions = self.config.output_attentions
        if rationale_targets is not None: output_attentions = True
        outputs = self.roberta(input_ids=input_ids,attention_mask=attention_mask,
            output_attentions=output_attentions,)

        sequence_output = outputs.last_hidden_state
        pooled_output = self.dropout(sequence_output[:, 0])
        logits = self.classifier(pooled_output).squeeze(-1)
        logits = torch.clamp(logits, -20.0, 20.0)

        if output_attentions:
            last_attentions = outputs.attentions[-1]
            cls_attentions = last_attentions.mean(dim=1)[:, 0, :]
            cls_attentions = cls_attentions * attention_mask.float()
            attn_sum = cls_attentions.sum(dim=-1, keepdim=True).clamp(min=1e-6)
            cls_attentions = cls_attentions / attn_sum
            cls_attentions = torch.nan_to_num(cls_attentions, nan=0.0)

        loss = None
        if labels is not None:
            cls_loss = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)(logits, labels.float())
            attn_loss = torch.tensor(0.0, device=logits.device)
            if rationale_targets is not None and output_attentions:
                tau = 1.0
                mask = (rationale_targets != -100) & (attention_mask == 1)
                num_tokens = mask.sum(dim=-1, keepdim=True).clamp(min=1)
                softened_targets = torch.zeros_like(rationale_targets)
                is_off = (labels == 1).unsqueeze(-1)
                uniform = (1.0 / num_tokens) * mask.float()
                rationale_soft = torch.softmax(rationale_targets / tau, dim=-1) * is_off.float()
                softened_targets = torch.where(is_off, rationale_soft, uniform)
                attn_loss = -(softened_targets * torch.log(cls_attentions + 1e-10)).sum(dim=-1) 
                attn_loss = (attn_loss * mask.sum(dim=-1) / num_tokens.squeeze(-1)).mean()
            lamda = 0.1
            loss = cls_loss + lamda * attn_loss
        attentions = outputs.attentions if output_attentions else None
        return SequenceClassifierOutput( loss=loss, logits=logits,hidden_states=outputs.hidden_states,
            attentions=attentions, )

In [7]:
gc.collect()
torch.cuda.empty_cache()

train_labels = np.array(train_dataset["labels"])
num_neg = np.sum(train_labels == 0)
num_pos = np.sum(train_labels == 1)
pos_weight_value = num_neg / num_pos if num_pos > 0 else 1.0
pos_weight_cpu = torch.tensor([pos_weight_value], dtype=torch.float32)

config = XLMRobertaConfig.from_pretrained(MODEL_NAME)
config.num_labels = 1
config.return_dict = True
config.output_attentions = False 
config.use_cache = False
model = RebertaExplain.from_pretrained(MODEL_NAME, config=config, attn_implementation="eager",
                                           use_safetensors=True)
model.config.output_attentions = False
model.config.use_cache = False
model.register_buffer("pos_weight", pos_weight_cpu)
model.to(device)

training_args = TrainingArguments( output_dir=LOCAL_OUT, num_train_epochs=N_EPOCHS,
    per_device_train_batch_size=2, per_device_eval_batch_size=1,
    gradient_accumulation_steps=8, evaluation_strategy="epoch",
    logging_steps=100,save_strategy="epoch",save_steps=100,
    save_total_limit=2,overwrite_output_dir=True,save_only_model=True,
    fp16=True,fp16_full_eval=True,bf16=False,bf16_full_eval=False,
    load_best_model_at_end=True,metric_for_best_model="f1_binary",
    greater_is_better=True,report_to="none",max_grad_norm=1.0,weight_decay=0.01,
    learning_rate=1e-5, warmup_ratio=0.1,lr_scheduler_type="linear",)


trainer = Trainer( model=model,args=training_args,train_dataset=train_dataset,
                  eval_dataset=val_dataset,compute_metrics=compute_metrics,)
trainer.train(resume_from_checkpoint="/workspace/lc_models/roberta/checkpoint-2165")

trainer.save_model(save_dir)        
tokenizer.save_pretrained(save_dir) 
print(f"✅ Model saved to {save_dir}")
gc.collect()
torch.cuda.empty_cache()

Some weights of RebertaExplain were not initialized from the model checkpoint at /workspace/models/xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pos_weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:2720: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the 

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Binary
2,0.736600,0.823814,0.859740,0.849549,0.810393
3,0.665800,0.903541,0.842597,0.834808,0.798938
4,0.630700,0.928563,0.858701,0.849375,0.811895


✅ Model saved to /workspace/ft_models/t1-m6-roberta-rat


In [ ]:
Epoch 	Training Loss 	Validation Loss 	Accuracy 	F1 Macro 	F1 Binary
0 	0.921600 	0.891830 	0.841039 	0.824620 	0.770958
1 	0.790800 	0.903351 	0.838961 	0.829869 	0.790541
2 	0.727500 	0.816377 	0.861299 	0.851605 	0.813678

In [5]:
config = XLMRobertaConfig.from_pretrained(save_dir)
config.output_attentions = False
config.use_cache = False
model = RebertaExplain.from_pretrained(save_dir, config=config,  ignore_mismatched_sizes=True, use_safetensors=True)
tokenizer = AutoTokenizer.from_pretrained(save_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

RebertaExplain(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [9]:
from torch.utils.data import DataLoader
CHUNK_SIZE = 256
test_loader = DataLoader(
    test_dataset,
    batch_size=CHUNK_SIZE,
    shuffle=False,
    num_workers=2,          # small speedup
    pin_memory=True         # faster GPU transfer
)

all_logits = []
model.eval()
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)     
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits   
        if logits.dim() == 2 and logits.size(1) == 1:
            logits = logits.squeeze(1)     
        all_logits.append(logits.detach().float().cpu().numpy())     
        del outputs, logits, input_ids, attention_mask
        gc.collect()
        torch.cuda.empty_cache()
all_logits = np.concatenate(all_logits, axis=0)
probs = 1.0 / (1.0 + np.exp(-all_logits))
preds = (probs >= 0.5).astype(int)
labels = np.array(test_dataset["labels"])
report = classification_report( labels, preds,labels=[0, 1],
    target_names=['NON', 'OFF'],
    output_dict=True,
    zero_division=0
)

print("Classification Report", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(report)
gc.collect()
torch.cuda.empty_cache()

timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
modelname = "T1-M6-roberta-rat"
with open(result_path, mode="a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["######################################################################"])
    writer.writerow([modelname, timestamp])
    writer.writerow(["Accuracy", r2(report.get("accuracy")), ])
    writer.writerow(["Class", "Precision", "Recall", "F1-Score", "Support"])
    for class_name, metrics in report.items():
        if isinstance(metrics, dict):
            writer.writerow([class_name,r2(metrics.get("precision")),r2(metrics.get("recall")),
                r2(metrics.get("f1-score")), r2(metrics.get("support")),])

Classification Report 2026-03-02 18:18:16
{'NON': {'precision': 0.9152196118488254, 'recall': 0.872444011684518, 'f1-score': 0.8933200398803589, 'support': 3081.0}, 'OFF': {'precision': 0.7905117270788913, 'recall': 0.8562355658198614, 'f1-score': 0.8220620842572062, 'support': 1732.0}, 'accuracy': 0.8666112611676708, 'macro avg': {'precision': 0.8528656694638583, 'recall': 0.8643397887521898, 'f1-score': 0.8576910620687825, 'support': 4813.0}, 'weighted avg': {'precision': 0.8703423925632393, 'recall': 0.8666112611676708, 'f1-score': 0.8676772434666251, 'support': 4813.0}}


In [6]:

def auprc(scores, y_true):
    from sklearn.metrics import average_precision_score
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    if len(y_true) == 0 or y_true.sum() == 0 or y_true.sum() == len(y_true):return 0.0
    return average_precision_score(y_true, scores)

def token_f1(y_pred, y_true):
    y_pred = np.asarray(y_pred)
    y_true = np.asarray(y_true)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    return f1

def iou(y_pred, y_true):
    pred_pos = np.where(y_pred == 1)[0]
    gold_pos = np.where(y_true == 1)[0]
    if len(pred_pos) == 0 or len(gold_pos) == 0:return 0.0
    intersection = len(set(pred_pos) & set(gold_pos))
    union = len(set(pred_pos) | set(gold_pos))
    return intersection / union if union > 0 else 0.0

def binarize_topk_nonzero(scores, k):
    scores = np.asarray(scores)
    nonzero_idx = np.where(scores > 0)[0]
    if len(nonzero_idx) == 0:return np.zeros(len(scores), dtype=int)
    k = min(k, len(nonzero_idx))
    top_k = nonzero_idx[np.argsort(scores[nonzero_idx])[-k:]]
    binary = np.zeros(len(scores), dtype=int)
    binary[top_k] = 1
    return binary

def mask_words(inputs, tokenizer, keep_word_mask, text):
    input_ids = inputs["input_ids"].clone() 
    attention_mask = inputs["attention_mask"].clone()    
    word_ids = inputs.get("word_ids")
    if word_ids is None:
        tokenized = tokenizer(text, return_offsets_mapping=True, return_tensors="pt")
        word_ids = tokenized.word_ids()   
    mask_token_id = tokenizer.mask_token_id or tokenizer.unk_token_id    
    current_word = -1
    for pos in range(1, input_ids.size(1)):
        if word_ids[pos] is None:continue
        if word_ids[pos] != current_word:
            current_word += 1
            if current_word >= len(keep_word_mask): break
            keep = keep_word_mask[current_word] 
        if not keep:input_ids[0, pos] = mask_token_id
    return {"input_ids": input_ids, "attention_mask": attention_mask}

def get_positive_probability(model, inputs, output_attentions=False):
    with torch.no_grad():
        model_inputs = {k: v for k, v in inputs.items() if k in ["input_ids", "attention_mask"]}
        outputs = model(**model_inputs, output_attentions=output_attentions)
        logits = outputs.logits.squeeze(-1)
        prob = torch.sigmoid(logits).clamp(0.0, 1.0).item()
    return prob

def subword_to_word_importance(token_scores, word_ids):
    if len(token_scores) == 0: return np.array([])
    word_importance = []
    current_word_id = None
    current = 0 
    input_ids = tokenized["input_ids"][0]
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    word_ids = tokenized.word_ids()
    offsets = tokenized["offset_mapping"][0]
    w = 0
    ws = 0
    for i, (tok, tid, wid, off) in enumerate(zip(tokens, input_ids, word_ids, offsets)):
        if wid is None:continue
        score = token_importance[i]
        if wid != w:
            #print(wid-1, ws)
            w = wid
            ws = score   
        else: ws += score
        #print(f"{i:3d} | {tok:12s} | id={tid.item():5d} | word_id={wid}") # ~| score = {score}| offset={tuple(off.tolist())}")
    for i, wid in enumerate(word_ids):
        if wid is None:continue
        if wid != current_word_id:
            if current_word_id is not None: word_importance.append(current)
            current_word_id = wid
            current = token_scores[i]
        else:
            current = current + token_scores[i]    
    if current_word_id is not None: word_importance.append(current)
    return np.array(word_importance)

def get_lime_importance(model,text,tokenizer,max_features=30, num_samples=2000):  
    def predict_proba(texts):
        enc = tokenizer(texts, padding=True,truncation=True,
            max_length=MAX_LENGTH, return_tensors="pt" ).to(device)
        with torch.no_grad():
            logits = model(**enc).logits
        logits = logits.squeeze(-1).unsqueeze(1)
        probs_pos = torch.sigmoid(logits).cpu().numpy()
        probs_neg = 1.0 - probs_pos
        probs = np.concatenate([probs_neg, probs_pos], axis=1)
        probs = np.clip(probs, 1e-6, 1 - 1e-6)
        return probs.astype(np.float32)    
    explainer = LimeTextExplainer( class_names=["NON", "OFF"],split_expression=r"\s+", bow=False,kernel_width=25, random_state=42)
    exp = explainer.explain_instance(text_instance=text, classifier_fn=predict_proba, num_features=max_features, 
                                     num_samples=num_samples, labels=(1,))
    lime_word_scores = dict(exp.as_list(label=1))
    words = text.split()
    word_importance = np.zeros(len(words), dtype=np.float32)
    for i, w in enumerate(words):
        if w in lime_word_scores: word_importance[i] = lime_word_scores[w]       
    return word_importance

In [10]:
results_attn = {"auprc": [], "token_f1": [], "iou": [], "comprehensiveness": [], "sufficiency": [] }
attn_miss = 0
REMOVE_CHARS = r"[|۔؟\?,\.،!_:\"\-_]"
for i, row in test_df[test_df[ycolumn] == 1].iterrows(): 
    text = row[xcolumn]
    text = re.sub(REMOVE_CHARS, "", text)
    gold_word_rationale = np.array(row["rationale"])
    if len(gold_word_rationale) == 0:continue
    tokenized = tokenizer(text,truncation=True, padding="max_length",max_length=MAX_LENGTH,return_offsets_mapping=True,return_tensors="pt" )
    word_ids = tokenized.word_ids()
    inputs = { "input_ids": tokenized["input_ids"].to(device),"attention_mask": tokenized["attention_mask"].to(device),"word_ids": word_ids}
    orig_prob = get_positive_probability(model, inputs, output_attentions=False)
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)
    attentions = outputs.attentions
    last_layer_attn = attentions[-1].squeeze(0)           
    avg_last_attn = last_layer_attn.mean(dim=0)        
    token_importance = avg_last_attn[0].cpu().numpy()  
    token_importance = np.maximum(token_importance, 0.0)
    token_importance[0] = 0.0
    word_importance = subword_to_word_importance(token_importance, word_ids)
    while len(gold_word_rationale) > len(word_importance):
        zeros = np.where(gold_word_rationale == 0)[0]
        if len(zeros) > 0: idx = zeros[0]
        else: idx = np.where(gold_word_rationale == 1)[0][0]
        gold_word_rationale = np.delete(gold_word_rationale, idx)

    while len(gold_word_rationale) < len(word_importance): 
         gold_word_rationale  = np.append(gold_word_rationale, 0)
    if len(word_importance) > len(gold_word_rationale):
        print("Length mismatch — skipping")
        attn_miss +=1
        continue
    k = min(8, len(word_importance))
    pred_word_binary = binarize_topk_nonzero(word_importance, k=k)
    results_attn["auprc"].append(auprc(word_importance, gold_word_rationale))
    results_attn["token_f1"].append(token_f1(pred_word_binary, gold_word_rationale))
    results_attn["iou"].append(iou(pred_word_binary, gold_word_rationale))
    comp_keep_mask = (pred_word_binary == 0)
    masked_comp_inputs = mask_words(inputs, tokenizer, comp_keep_mask, text)
    comp_prob = get_positive_probability(model, masked_comp_inputs, output_attentions=False)
    results_attn["comprehensiveness"].append(orig_prob - comp_prob)
    suff_keep_mask = (pred_word_binary == 1)
    masked_suff_inputs = mask_words(inputs, tokenizer, suff_keep_mask, text)
    suff_prob = get_positive_probability(model, masked_suff_inputs, output_attentions=False)
    results_attn["sufficiency"].append(orig_prob -suff_prob)  

print("\nDataset-level results (macro average over positive examples):\n")
print("attn_miss:",attn_miss)
for name, values in results_attn.items():
    if not values:
        print(f"{name:18}: no valid examples")
        continue
    mean = np.mean(values)
    std  = np.std(values)
    n    = len(values)
    print(f"{name:18}: {mean:.4f} ± {std:.4f}  (n={n})")

timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
with open(result_path, mode="a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["--- ATTENTION RESULTS ---", timestamp])
    for metric_name, values in results_attn.items():
        mean_val = np.mean(values)
        std_val  = np.std(values)
        writer.writerow([metric_name,"-ATTN--",r2(mean_val),r2(std_val)])
print("Result Saved")


Dataset-level results (macro average over positive examples):

attn_miss: 0
auprc             : 0.6072 ± 0.2738  (n=1732)
token_f1          : 0.4477 ± 0.2176  (n=1732)
iou               : 0.3159 ± 0.1997  (n=1732)
comprehensiveness : 0.6568 ± 0.3835  (n=1732)
sufficiency       : 0.0561 ± 0.2670  (n=1732)
Result Saved


In [ ]:
results_lime = { "auprc": [],"token_f1": [], "iou": [],"comprehensiveness": [], "sufficiency": []}
start_idx = 0
try:
    existing_df = pd.read_csv(SAVE_result)
    if not existing_df.empty:
        for col in results_lime.keys():
            if col in existing_df.columns:
                results_lime[col] = existing_df[col].tolist()       
        start_idx = len(results_lime["auprc"])  # number already processed
        print(f"Resuming from row {start_idx} (already processed {start_idx} examples)")
        print(f"Current datetime: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    else:
        print("CSV exists but is empty → starting from scratch")
except FileNotFoundError:
    print("No existing CSV found → starting from scratch")
except Exception as e:
    print(f"Error loading CSV: {e} → starting from scratch")
i = start_idx 
positive_rows = test_df[test_df[ycolumn] == 1].reset_index(drop=True)
remaining_rows = positive_rows.iloc[start_idx:]
print(f"Total positive rows: {len(positive_rows)}")
print(f"Already processed: {start_idx}")
print(f"Remaining to process: {len(remaining_rows)}")

for idx, row in remaining_rows.iterrows():
    if i % 100 == 0:
        print(f"Processed {i} / {len(positive_rows)}, saving full results... "
              f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        pd.DataFrame(results_lime).to_csv(SAVE_result, index=False)
    i += 1
    text = row[xcolumn]
    gold_word_rationale = np.array(row["rationale"])
    if len(gold_word_rationale) == 0:continue
    tokenized = tokenizer( text, truncation=True,padding="max_length", max_length=MAX_LENGTH,return_offsets_mapping=True,return_tensors="pt")
    word_ids = tokenized.word_ids()
    inputs = { "input_ids": tokenized["input_ids"].to(device), "attention_mask": tokenized["attention_mask"].to(device), "word_ids": word_ids}
    lime_scores = get_lime_importance( model=model, text=text, tokenizer=tokenizer,  
                                      max_features=len(gold_word_rationale), num_samples=2000)
    word_importance = np.maximum(lime_scores, 0.0)
    word_importance = word_importance / (word_importance.sum() + 1e-12)  
    orig_prob = get_positive_probability(model, inputs, output_attentions=False)
    # Check alignment with gold rationale length
    if len(word_importance) != len(gold_word_rationale):
        print(f"Length mismatch (example {idx}) — skipping")
        continue
    k = min(8, len(word_importance))
    pred_word_binary = binarize_topk_nonzero(word_importance, k=k)
    results_lime["auprc"].append(auprc(word_importance, gold_word_rationale))
    results_lime["token_f1"].append(token_f1(pred_word_binary, gold_word_rationale))
    results_lime["iou"].append(iou(pred_word_binary, gold_word_rationale))
    # Comprehensiveness: keep NON-rationale words
    comp_keep_mask = (pred_word_binary == 0)
    masked_comp_inputs = mask_words(inputs, tokenizer, comp_keep_mask, text)
    comp_prob = get_positive_probability(model, masked_comp_inputs, output_attentions=False)
    results_lime["comprehensiveness"].append(orig_prob - comp_prob)
    # Sufficiency: keep ONLY rationale words
    suff_keep_mask = (pred_word_binary == 1)
    masked_suff_inputs = mask_words(inputs, tokenizer, suff_keep_mask, text)
    suff_prob = get_positive_probability(model, masked_suff_inputs, output_attentions=False)
    results_lime["sufficiency"].append(orig_prob - suff_prob)

# Final save after loop completes
pd.DataFrame(results_lime).to_csv(SAVE_result, index=False)
print(f"Finished! Total processed: {len(results_lime['auprc'])}")
print(f"Results saved to: {SAVE_result}")
print(f"Final datetime: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print("\nLIME-based Dataset-level results (macro average over positive examples):\n")
for name, values in results_lime.items():
    if not values:
        print(f"{name:18}: no valid examples")
        continue
    mean = np.mean(values)
    std  = np.std(values)
    n    = len(values)
    print(f"{name:18}: {mean:.4f} ± {std:.4f}  (n={n})")

with open(result_path, mode="a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["--- LIME RESULTS ---"])
    for metric_name, values in results_lime.items():
        mean_val = np.mean(values)
        std_val  = np.std(values)
        writer.writerow([metric_name,"-LIME--",r2(mean_val),r2(std_val)])
print("File Saved")

No existing CSV found → starting from scratch
Total positive rows: 1732
Already processed: 0
Remaining to process: 1732
Processed 0 / 1732, saving full results... 2026-03-04 07:36:37
Processed 100 / 1732, saving full results... 2026-03-04 07:42:25
Processed 200 / 1732, saving full results... 2026-03-04 07:48:03
Processed 300 / 1732, saving full results... 2026-03-04 07:53:43
Processed 400 / 1732, saving full results... 2026-03-04 07:59:32
Processed 500 / 1732, saving full results... 2026-03-04 08:05:05
Processed 600 / 1732, saving full results... 2026-03-04 08:10:44
Processed 700 / 1732, saving full results... 2026-03-04 08:16:26
Processed 800 / 1732, saving full results... 2026-03-04 08:21:39
Processed 900 / 1732, saving full results... 2026-03-04 08:27:14
Processed 1000 / 1732, saving full results... 2026-03-04 08:32:50
Processed 1100 / 1732, saving full results... 2026-03-04 08:38:15
Processed 1200 / 1732, saving full results... 2026-03-04 08:44:11
Processed 1300 / 1732, saving full

In [ ]:
script_name = os.path.basename(__file__) if "__file__" in globals() else "interactive"
timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("######################################################################")
print(f"Model: {modelname} | Script: {script_name} | Timestamp: {timestamp}\n")
print('Accuracy:',r2(report["accuracy"]))
print(f"{'Class':<15} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'Support':<10}")
for class_name, metrics in report.items():
    if isinstance(metrics, dict):
        print(f"{class_name:<15} {r2(metrics['precision']):<10} {r2(metrics['recall']):<10} "
              f"{r2(metrics['f1-score']):<10} {r2(metrics['support']):<10}")
# Token-level / Faithfulness metrics
print("\nToken-level / Faithfulness Metrics (Mean ± Std):")
print(f"{'Metric':<20} {'Attention':<15} {'LIME':<15}")
for metric_name in results_attn.keys():
    attn_mean = r2(np.mean(results_attn[metric_name]))
    attn_std = r2(np.std(results_attn[metric_name]))
    lime_mean = r2(np.mean(results_lime[metric_name]))
    lime_std = r2(np.std(results_lime[metric_name]))
    print(f"{metric_name:<20} {attn_mean} ± {attn_std:<11} {lime_mean} ± {lime_std:<11}")
